In [ ]:
import pandas as pd
import numpy as np
from statsbombpy import sb
import time

# List of major tournaments to scrape (competition_id, season_id)
# Feel free to add more tournaments here later
TOURNAMENTS = [
    {"name": "World Cup 2022", "comp_id": 43, "season_id": 106},
    {"name": "World Cup 2018", "comp_id": 43, "season_id": 3},
    {"name": "Euro 2024",      "comp_id": 55, "season_id": 282},
    {"name": "Euro 2020",      "comp_id": 55, "season_id": 43},
    {"name": "Copa America 2024", "comp_id": 223, "season_id": 282}
]

def extract_tournament_data(comp_id, season_id, tournament_name):
    print(f"\n---> Fetching match list for {tournament_name}...")

    try:
        matches = sb.matches(competition_id=comp_id, season_id=season_id)
    except Exception as e:
        print(f"Failed to fetch {tournament_name}. Error: {e}")
        return []

    print(f"Found {len(matches)} matches. Extracting stats now...")
    match_rows = []

    for idx, match in matches.iterrows():
        m_id = match['match_id']
        home_team = match['home_team']
        away_team = match['away_team']
        score_str = f"{match['home_score']}-{match['away_score']}"

        # Pull event data for the match
        try:
            events = sb.events(match_id=m_id)
        except Exception:
            continue # Skip if event data is missing or API times out

        # Split events by team
        home_events = events[events['team'] == home_team]
        away_events = events[events['team'] == away_team]

        # 1. Expected Goals (xG)
        home_xg = home_events['shot_statsbomb_xg'].sum() if 'shot_statsbomb_xg' in home_events.columns else 0.0
        away_xg = away_events['shot_statsbomb_xg'].sum() if 'shot_statsbomb_xg' in away_events.columns else 0.0

        # 2. Shots & Shots on Target
        home_shots = home_events[home_events['type'] == 'Shot'] if 'type' in home_events.columns else pd.DataFrame()
        away_shots = away_events[away_events['type'] == 'Shot'] if 'type' in away_events.columns else pd.DataFrame()

        home_sot = len(home_shots[home_shots['shot_outcome'].isin(['Goal', 'Saved', 'Post'])]) if 'shot_outcome' in home_shots.columns else 0
        away_sot = len(away_shots[away_shots['shot_outcome'].isin(['Goal', 'Saved', 'Post'])]) if 'shot_outcome' in away_shots.columns else 0

        # 3. Passes & Possession
        home_passes = home_events[home_events['type'] == 'Pass'] if 'type' in home_events.columns else pd.DataFrame()
        away_passes = away_events[away_events['type'] == 'Pass'] if 'type' in away_events.columns else pd.DataFrame()

        home_pass_complete = len(home_passes[home_passes['pass_outcome'].isna()]) if 'pass_outcome' in home_passes.columns else len(home_passes)
        away_pass_complete = len(away_passes[away_passes['pass_outcome'].isna()]) if 'pass_outcome' in away_passes.columns else len(away_passes)

        total_passes = max(len(home_passes) + len(away_passes), 1)
        home_poss = round((len(home_passes) / total_passes) * 100, 1)
        away_poss = round((len(away_passes) / total_passes) * 100, 1)

        # 4. Defending (Tackles & Interceptions for PPDA calculation later)
        home_tackles = len(home_events[home_events['type'] == 'Duel']) if 'type' in home_events.columns else 0
        away_tackles = len(away_events[away_events['type'] == 'Duel']) if 'type' in away_events.columns else 0

        home_interceptions = len(home_events[home_events['type'] == 'Interception']) if 'type' in home_events.columns else 0
        away_interceptions = len(away_events[away_events['type'] == 'Interception']) if 'type' in away_events.columns else 0

        # 5. Aerial Duels
        home_aerials = len(home_events[home_events['type'] == '50/50']) if 'type' in home_events.columns else 1
        away_aerials = len(away_events[away_events['type'] == '50/50']) if 'type' in away_events.columns else 1

        # Build the row dictionary
        # Using default values (e.g., 5 or 10) for irrelevant columns
        # to keep the schema strictly compatible with the old data.csv
        match_rows.append({
            'match': f"{home_team} vs {away_team}",
            'tournament': tournament_name,
            'score': score_str,
            'home_team': home_team,
            'away_team': away_team,
            'home_xg': round(home_xg, 2),
            'away_xg': round(away_xg, 2),
            'home_possession': home_poss,
            'away_possession': away_poss,
            'home_sot': home_sot,
            'away_sot': away_sot,
            'home_total_shots': len(home_shots),
            'away_total_shots': len(away_shots),
            'home_completed_passes': home_pass_complete,
            'away_completed_passes': away_pass_complete,
            'home_attempted_pases': len(home_passes),
            'away_attempted_pases': len(away_passes),
            'home_tackles': home_tackles,
            'away_tackles': away_tackles,
            'home_interceptions': home_interceptions,
            'away_interceptions': away_interceptions,
            'home_clearances': 10, # Dummy value, skipped by model
            'away_clearances': 10,
            'home_fouls': 10,      # Dummy value
            'away_fouls': 10,
            'home_corners': 5,     # Dummy value
            'away_corners': 5,
            'home_crosses': 10,    # Dummy value
            'away_crosses': 10,
            'home_aerials_won': home_aerials,
            'away_aerials_won': away_aerials
        })
    return match_rows

all_matches = []

# Loop through each tournament and aggregate the data
for tourn in TOURNAMENTS:
    data = extract_tournament_data(tourn['comp_id'], tourn['season_id'], tourn['name'])
    all_matches.extend(data)

# Save everything into a single CSV ready for the training pipeline
df_all = pd.DataFrame(all_matches)
df_all.to_csv('data.csv', index=False)

print(f"\nDone! Saved {len(df_all)} matches to data.csv.")


---> Fetching match list for World Cup 2022...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Found 64 matches. Extracting stats now...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: 


---> Fetching match list for World Cup 2018...


...


---> Fetching match list for Copa America 2024...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Found 32 matches. Extracting stats now...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:27: 


Done! Saved 262 matches to data.csv.
